In [1]:
import sys
import os

# 切换到项目根目录（如果 notebook 在其他位置）
os.chdir(r'C:\Users\king\Desktop\tabular_dl_project')
# 将当前目录加入 sys.path
sys.path.insert(0, os.path.abspath('.'))

In [2]:
import sys
import os

# 切换到项目根目录（根据你的实际路径）
project_root = r'C:\Users\king\Desktop\tabular_dl_project'
os.chdir(project_root)
sys.path.insert(0, project_root)

# 验证当前工作目录
print("当前工作目录:", os.getcwd())

当前工作目录: c:\Users\king\Desktop\tabular_dl_project


In [3]:
import sys, os

# 1. 查看当前工作目录
print("当前工作目录:", os.getcwd())

# 2. 期望的项目根目录
project_root = r'C:\Users\king\Desktop\tabular_dl_project'
print("项目根目录:", project_root)

# 3. 检查文件是否存在
ft_file = os.path.join(project_root, 'src', 'models', 'ft_transformer.py')
init_file = os.path.join(project_root, 'src', 'models', '__init__.py')
print("ft_transformer.py 存在?", os.path.exists(ft_file))
print("models/__init__.py 存在?", os.path.exists(init_file))

# 4. 查看 sys.path 前几个路径
print("sys.path 前3项:", sys.path[:3])

当前工作目录: c:\Users\king\Desktop\tabular_dl_project
项目根目录: C:\Users\king\Desktop\tabular_dl_project
ft_transformer.py 存在? True
models/__init__.py 存在? True
sys.path 前3项: ['C:\\Users\\king\\Desktop\\tabular_dl_project', 'c:\\Users\\king\\Desktop\\tabular_dl_project', 'C:\\Users\\king\\AppData\\Local\\Programs\\Python\\Python312\\python312.zip']


In [4]:
import os
path = r'C:\Users\king\Desktop\tabular_dl_project\src\models'
print("存在?", os.path.exists(path))
print("是目录?", os.path.isdir(path))

存在? True
是目录? True


In [5]:
import os

project_root = r'C:\Users\king\Desktop\tabular_dl_project'
models_path = os.path.join(project_root, 'src', 'models')

# 如果是文件则删除
if os.path.exists(models_path) and not os.path.isdir(models_path):
    os.remove(models_path)
    print("已删除文件 models")

# 创建目录
os.makedirs(models_path, exist_ok=True)
print("目录 models 创建成功")

# 创建 __init__.py
init_path = os.path.join(models_path, '__init__.py')
with open(init_path, 'w') as f:
    pass
print("Created models/__init__.py")

# 创建 ft_transformer.py（代码复用之前的 ft_code）
ft_code = '''import torch
import torch.nn as nn

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64):
        super().__init__()
        self.n_num_features = n_num_features
        self.cat_cardinalities = cat_cardinalities
        if n_num_features > 0:
            self.num_linear = nn.Linear(n_num_features, embedding_dim)
        else:
            self.num_linear = None
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(card, embedding_dim) for card in cat_cardinalities
        ])
    def forward(self, x_num, x_cat):
        tokens = []
        if self.num_linear is not None:
            tokens.append(self.num_linear(x_num))
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i].long()))
        return torch.stack(tokens, dim=1)

class FTTransformer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64,
                 n_blocks=3, n_heads=8, ff_dim=128, dropout=0.1, output_dim=1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, cat_cardinalities, embedding_dim)
        n_tokens = (1 if n_num_features > 0 else 0) + len(cat_cardinalities)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.output_layer = nn.Linear(embedding_dim, output_dim)
    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)
        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = self.transformer(tokens)
        cls_out = tokens[:, 0, :]
        return self.output_layer(cls_out).squeeze(-1)
'''

ft_path = os.path.join(models_path, 'ft_transformer.py')
with open(ft_path, 'w') as f:
    f.write(ft_code)
print("Created ft_transformer.py")

# 验证导入
from src.models.ft_transformer import FTTransformer
print("导入成功！")

目录 models 创建成功
Created models/__init__.py
Created ft_transformer.py
导入成功！


In [6]:
import os

project_root = r'C:\Users\king\Desktop\tabular_dl_project'

# 创建 models 目录和 __init__.py
os.makedirs(os.path.join(project_root, 'src', 'models'), exist_ok=True)
init_path = os.path.join(project_root, 'src', 'models', '__init__.py')
if not os.path.exists(init_path):
    with open(init_path, 'w') as f:
        f.write('')
    print("Created models/__init__.py")

# 创建 ft_transformer.py
ft_path = os.path.join(project_root, 'src', 'models', 'ft_transformer.py')
ft_code = '''import torch
import torch.nn as nn

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64):
        super().__init__()
        self.n_num_features = n_num_features
        self.cat_cardinalities = cat_cardinalities
        if n_num_features > 0:
            self.num_linear = nn.Linear(n_num_features, embedding_dim)
        else:
            self.num_linear = None
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(card, embedding_dim) for card in cat_cardinalities
        ])
    def forward(self, x_num, x_cat):
        tokens = []
        if self.num_linear is not None:
            tokens.append(self.num_linear(x_num))
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i].long()))
        return torch.stack(tokens, dim=1)

class FTTransformer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64,
                 n_blocks=3, n_heads=8, ff_dim=128, dropout=0.1, output_dim=1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, cat_cardinalities, embedding_dim)
        n_tokens = (1 if n_num_features > 0 else 0) + len(cat_cardinalities)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.output_layer = nn.Linear(embedding_dim, output_dim)
    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)
        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = self.transformer(tokens)
        cls_out = tokens[:, 0, :]
        return self.output_layer(cls_out).squeeze(-1)
'''

with open(ft_path, 'w') as f:
    f.write(ft_code)
print("Created ft_transformer.py")

# 验证导入
from src.models.ft_transformer import FTTransformer
print("导入成功！")

Created ft_transformer.py
导入成功！


In [7]:
from src.models.ft_transformer import FTTransformer
print("导入成功")

导入成功


In [8]:
import random
import numpy as np
import torch
import os
import json
from typing import Any, Dict

def set_seed(seed: int = 42):
    """设置所有随机种子以保证可复现性"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

def save_results(results: Dict[str, Any], filename: str):
    """保存结果到 JSON 文件"""
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w') as f:
        json.dump(results, f, indent=4)

def load_results(filename: str) -> Dict[str, Any]:
    with open(filename, 'r') as f:
        return json.load(f)

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import os

def load_adult_data(data_path='data/raw/adult.data'):
    """加载 Adult 数据集，返回原始 DataFrame 和标签"""
    column_names = [
        'age', 'workclass', 'fnlwgt', 'education', 'education_num',
        'marital_status', 'occupation', 'relationship', 'race', 'sex',
        'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
    ]
    df = pd.read_csv(data_path, names=column_names, na_values='?', skipinitialspace=True)
    # 删除包含缺失值的行（简单处理，也可插补）
    df = df.dropna().reset_index(drop=True)
    # 标签：二分类，>50K 为 1
    y = (df['income'] == '>50K').astype(int)
    X = df.drop('income', axis=1)
    return X, y

def get_feature_types(X):
    """自动识别数值列和类别列"""
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    return num_cols, cat_cols

def split_data(X, y, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, random_state=42):
    """划分训练/验证/测试集，返回索引和分割后的数据"""
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_ratio, random_state=random_state, stratify=y
    )
    val_relative = val_ratio / (train_ratio + val_ratio)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_relative, random_state=random_state, stratify=y_temp
    )
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def preprocess_for_tree(X_train, X_val, X_test, cat_cols):
    """树模型：类别特征采用 Label Encoding，缺失值保留为 -1 或由模型处理"""
    X_train_tree = X_train.copy()
    X_val_tree = X_val.copy()
    X_test_tree = X_test.copy()
    le_dict = {}
    for col in cat_cols:
        le = LabelEncoder()
        # 合并训练+验证集来拟合，避免测试集出现新类别
        combined = pd.concat([X_train_tree[col], X_val_tree[col]], axis=0).astype(str)
        le.fit(combined)
        X_train_tree[col] = le.transform(X_train_tree[col].astype(str))
        X_val_tree[col] = le.transform(X_val_tree[col].astype(str))
        X_test_tree[col] = le.transform(X_test_tree[col].astype(str))
        le_dict[col] = le
    # 树模型不需要标准化，但缺失值保留（XGBoost/LightGBM 支持 NaN）
    return X_train_tree, X_val_tree, X_test_tree, le_dict

def preprocess_for_linear(X_train, X_val, X_test, num_cols, cat_cols):
    """线性模型：One-Hot 编码类别特征 + 标准化数值特征，缺失值用均值插补"""
    from sklearn.preprocessing import OneHotEncoder
    # 处理数值特征：插补 + 标准化
    num_imputer = SimpleImputer(strategy='median')
    X_train_num = num_imputer.fit_transform(X_train[num_cols])
    X_val_num = num_imputer.transform(X_val[num_cols])
    X_test_num = num_imputer.transform(X_test[num_cols])
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num)
    X_val_num = scaler.transform(X_val_num)
    X_test_num = scaler.transform(X_test_num)
    
    # 处理类别特征：One-Hot
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_train_cat = ohe.fit_transform(X_train[cat_cols].astype(str))
    X_val_cat = ohe.transform(X_val[cat_cols].astype(str))
    X_test_cat = ohe.transform(X_test[cat_cols].astype(str))
    
    # 合并
    X_train_lr = np.hstack([X_train_num, X_train_cat])
    X_val_lr = np.hstack([X_val_num, X_val_cat])
    X_test_lr = np.hstack([X_test_num, X_test_cat])
    return X_train_lr, X_val_lr, X_test_lr, scaler, ohe

def preprocess_for_dl(X_train, X_val, X_test, num_cols, cat_cols):
    """深度学习：数值标准化，类别用 Label Encoding 并返回类别数（用于 Embedding）"""
    # 数值：插补 + 标准化
    num_imputer = SimpleImputer(strategy='median')
    X_train_num = num_imputer.fit_transform(X_train[num_cols])
    X_val_num = num_imputer.transform(X_val[num_cols])
    X_test_num = num_imputer.transform(X_test[num_cols])
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num)
    X_val_num = scaler.transform(X_val_num)
    X_test_num = scaler.transform(X_test_num)
    
    # 类别：Label Encoding（保持与树模型类似，但记录类别数用于 embedding）
    cat_encoders = {}
    cat_cardinalities = []
    X_train_cat = []
    X_val_cat = []
    X_test_cat = []
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_val[col]], axis=0).astype(str)
        le.fit(combined)
        X_train_cat.append(le.transform(X_train[col].astype(str)))
        X_val_cat.append(le.transform(X_val[col].astype(str)))
        X_test_cat.append(le.transform(X_test[col].astype(str)))
        cat_encoders[col] = le
        cat_cardinalities.append(len(le.classes_))
    X_train_cat = np.column_stack(X_train_cat) if cat_cols else np.empty((len(X_train), 0))
    X_val_cat = np.column_stack(X_val_cat) if cat_cols else np.empty((len(X_val), 0))
    X_test_cat = np.column_stack(X_test_cat) if cat_cols else np.empty((len(X_test), 0))
    
    return (X_train_num, X_train_cat), (X_val_num, X_val_cat), (X_test_num, X_test_cat), scaler, cat_cardinalities

In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64):
        super().__init__()
        self.n_num_features = n_num_features
        self.cat_cardinalities = cat_cardinalities
        self.embedding_dim = embedding_dim
        
        if n_num_features > 0:
            self.num_linear = nn.Linear(n_num_features, embedding_dim)
        else:
            self.num_linear = None
        
        self.cat_embeddings = nn.ModuleList([
            nn.Embedding(card, embedding_dim) for card in cat_cardinalities
        ])
    
    def forward(self, x_num, x_cat):
        # x_num: (batch, n_num), x_cat: (batch, n_cat)
        tokens = []
        if self.num_linear is not None:
            tokens.append(self.num_linear(x_num))
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i].long()))
        # tokens: list of (batch, emb_dim) -> stack to (batch, n_tokens, emb_dim)
        return torch.stack(tokens, dim=1)

class FTTransformer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64,
                 n_blocks=3, n_heads=8, ff_dim=128, dropout=0.1, output_dim=1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, cat_cardinalities, embedding_dim)
        n_tokens = (1 if n_num_features > 0 else 0) + len(cat_cardinalities)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, activation='gelu', batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.output_layer = nn.Linear(embedding_dim, output_dim)
    
    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)  # (B, T, D)
        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)  # (B, 1+T, D)
        tokens = self.transformer(tokens)
        cls_out = tokens[:, 0, :]  # (B, D)
        logits = self.output_layer(cls_out)  # (B, output_dim)
        return logits.squeeze(-1) if output_dim == 1 else logits

In [11]:
import os

project_root = r'C:\Users\king\Desktop\tabular_dl_project'
utils_path = os.path.join(project_root, 'src', 'utils.py')

utils_code = '''import random
import numpy as np
import torch
import os
import json
from typing import Any, Dict

def set_seed(seed: int = 42):
    \"\"\"设置所有随机种子以保证可复现性\"\"\"
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

def save_results(results: Dict[str, Any], filename: str):
    \"\"\"保存结果到 JSON 文件\"\"\"
    os.makedirs(os.path.dirname(filename), exist_ok=True)
    with open(filename, 'w') as f:
        json.dump(results, f, indent=4)

def load_results(filename: str) -> Dict[str, Any]:
    with open(filename, 'r') as f:
        return json.load(f)
'''

with open(utils_path, 'w', encoding='utf-8') as f:
    f.write(utils_code)
print("Created src/utils.py with UTF-8 encoding")

Created src/utils.py with UTF-8 encoding


In [12]:
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import joblib
from src.models.ft_transformer import FTTransformer
from src.utils import set_seed

def train_xgboost(X_train, y_train, X_val, y_val, params, seed=42):
    model = XGBClassifier(random_state=seed, eval_metric='logloss', **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model

def train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    model = LGBMClassifier(random_state=seed, **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[], verbose=False)
    return model

def train_logistic(X_train, y_train, X_val, y_val, params, seed=42):
    model = LogisticRegression(random_state=seed, max_iter=1000, **params)
    model.fit(X_train, y_train)
    return model

def train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    from pytorch_tabnet.tab_model import TabNetClassifier
    model = TabNetClassifier(seed=seed, **params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=['auc'],
        max_epochs=50,
        patience=10,
        batch_size=256,
        virtual_batch_size=128,
        verbose=0
    )
    return model

def train_ft_transformer(X_num_train, X_cat_train, y_train,
                         X_num_val, X_cat_val, y_val,
                         cat_cardinalities, params, seed=42):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FTTransformer(
        n_num_features=X_num_train.shape[1],
        cat_cardinalities=cat_cardinalities,
        embedding_dim=params.get('embedding_dim', 64),
        n_blocks=params.get('n_blocks', 3),
        n_heads=params.get('n_heads', 8),
        ff_dim=params.get('ff_dim', 128),
        dropout=params.get('dropout', 0.1),
        output_dim=1
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=params.get('lr', 1e-3))
    criterion = nn.BCEWithLogitsLoss()
    
    train_dataset = TensorDataset(torch.FloatTensor(X_num_train), torch.LongTensor(X_cat_train), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_num_val), torch.LongTensor(X_cat_val), torch.FloatTensor(y_val))
    train_loader = DataLoader(train_dataset, batch_size=params.get('batch_size', 256), shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256)
    
    best_val_auc = 0
    best_model_state = None
    patience = 10
    epochs_no_improve = 0
    for epoch in range(params.get('epochs', 100)):
        model.train()
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x_num, x_cat)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        # validation
        model.eval()
        y_pred_list = []
        y_true_list = []
        with torch.no_grad():
            for x_num, x_cat, y in val_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_list.append(probs.cpu().numpy())
                y_true_list.append(y.cpu().numpy())
        y_pred = np.concatenate(y_pred_list)
        y_true = np.concatenate(y_true_list)
        val_auc = roc_auc_score(y_true, y_pred)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            break
    model.load_state_dict(best_model_state)
    return model

def evaluate_model(model, X_test, y_test, model_type, cat_cardinalities=None, X_cat_test=None):
    """评估模型：预测、AUC、Accuracy、F1、推理延迟（毫秒/样本）、模型大小（MB）"""
    # 推理延迟
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        def predict_proba_batch(X):
            if model_type == 'logistic':
                return model.predict_proba(X)[:, 1]
            else:
                return model.predict_proba(X)[:, 1]
        # 测量延迟（1000样本，10次平均）
        sample_X = X_test[:1000]
        start = time.perf_counter()
        for _ in range(10):
            _ = predict_proba_batch(sample_X)
        latency = (time.perf_counter() - start) / (10 * len(sample_X)) * 1000  # ms per sample
        y_pred_proba = predict_proba_batch(X_test)
    elif model_type == 'tabnet':
        start = time.perf_counter()
        for _ in range(10):
            _ = model.predict_proba(X_test[:1000])[:, 1]
        latency = (time.perf_counter() - start) / (10 * 1000) * 1000
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif model_type == 'ft_transformer':
        device = next(model.parameters()).device
        model.eval()
        X_num_test, X_cat_test = X_test
        test_dataset = TensorDataset(torch.FloatTensor(X_num_test), torch.LongTensor(X_cat_test))
        test_loader = DataLoader(test_dataset, batch_size=256)
        y_pred_proba_list = []
        start = time.perf_counter()
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_proba_list.append(probs.cpu().numpy())
        latency = (time.perf_counter() - start) / len(X_num_test) * 1000
        y_pred_proba = np.concatenate(y_pred_proba_list)
    
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    # 模型大小（MB）
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        import tempfile
        with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
            joblib.dump(model, f.name)
            size_mb = os.path.getsize(f.name) / (1024 * 1024)
            os.unlink(f.name)
    elif model_type == 'tabnet':
        import tempfile
        with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as f:
            model.save_model(f.name)
            size_mb = os.path.getsize(f.name) / (1024 * 1024)
            os.unlink(f.name)
    else:
        import tempfile
        with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
            torch.save(model.state_dict(), f.name)
            size_mb = os.path.getsize(f.name) / (1024 * 1024)
            os.unlink(f.name)
    
    return {'auc': auc, 'accuracy': acc, 'f1': f1, 'latency_ms_per_sample': latency, 'model_size_mb': size_mb}

In [13]:
import os

project_root = r'C:\Users\king\Desktop\tabular_dl_project'

# 确保 data/raw 目录存在（用于存放数据集）
os.makedirs(os.path.join(project_root, 'data', 'raw'), exist_ok=True)

# 1. preprocess.py
preprocess_code = '''import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

def load_adult_data(data_path='data/raw/adult.data'):
    column_names = [
        'age', 'workclass', 'fnlwgt', 'education', 'education_num',
        'marital_status', 'occupation', 'relationship', 'race', 'sex',
        'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
    ]
    df = pd.read_csv(data_path, names=column_names, na_values='?', skipinitialspace=True)
    df = df.dropna().reset_index(drop=True)
    y = (df['income'] == '>50K').astype(int)
    X = df.drop('income', axis=1)
    return X, y

def get_feature_types(X):
    num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
    cat_cols = X.select_dtypes(include=['object']).columns.tolist()
    return num_cols, cat_cols

def split_data(X, y, train_ratio=0.6, val_ratio=0.2, test_ratio=0.2, random_state=42):
    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_ratio, random_state=random_state, stratify=y
    )
    val_relative = val_ratio / (train_ratio + val_ratio)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_relative, random_state=random_state, stratify=y_temp
    )
    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

def preprocess_for_tree(X_train, X_val, X_test, cat_cols):
    X_train_tree = X_train.copy()
    X_val_tree = X_val.copy()
    X_test_tree = X_test.copy()
    le_dict = {}
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train_tree[col], X_val_tree[col]], axis=0).astype(str)
        le.fit(combined)
        X_train_tree[col] = le.transform(X_train_tree[col].astype(str))
        X_val_tree[col] = le.transform(X_val_tree[col].astype(str))
        X_test_tree[col] = le.transform(X_test_tree[col].astype(str))
        le_dict[col] = le
    return X_train_tree, X_val_tree, X_test_tree, le_dict

def preprocess_for_linear(X_train, X_val, X_test, num_cols, cat_cols):
    num_imputer = SimpleImputer(strategy='median')
    X_train_num = num_imputer.fit_transform(X_train[num_cols])
    X_val_num = num_imputer.transform(X_val[num_cols])
    X_test_num = num_imputer.transform(X_test[num_cols])
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num)
    X_val_num = scaler.transform(X_val_num)
    X_test_num = scaler.transform(X_test_num)
    
    from sklearn.preprocessing import OneHotEncoder
    ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    X_train_cat = ohe.fit_transform(X_train[cat_cols].astype(str))
    X_val_cat = ohe.transform(X_val[cat_cols].astype(str))
    X_test_cat = ohe.transform(X_test[cat_cols].astype(str))
    
    X_train_lr = np.hstack([X_train_num, X_train_cat])
    X_val_lr = np.hstack([X_val_num, X_val_cat])
    X_test_lr = np.hstack([X_test_num, X_test_cat])
    return X_train_lr, X_val_lr, X_test_lr, scaler, ohe

def preprocess_for_dl(X_train, X_val, X_test, num_cols, cat_cols):
    num_imputer = SimpleImputer(strategy='median')
    X_train_num = num_imputer.fit_transform(X_train[num_cols])
    X_val_num = num_imputer.transform(X_val[num_cols])
    X_test_num = num_imputer.transform(X_test[num_cols])
    scaler = StandardScaler()
    X_train_num = scaler.fit_transform(X_train_num)
    X_val_num = scaler.transform(X_val_num)
    X_test_num = scaler.transform(X_test_num)
    
    cat_encoders = {}
    cat_cardinalities = []
    X_train_cat = []
    X_val_cat = []
    X_test_cat = []
    for col in cat_cols:
        le = LabelEncoder()
        combined = pd.concat([X_train[col], X_val[col]], axis=0).astype(str)
        le.fit(combined)
        X_train_cat.append(le.transform(X_train[col].astype(str)))
        X_val_cat.append(le.transform(X_val[col].astype(str)))
        X_test_cat.append(le.transform(X_test[col].astype(str)))
        cat_encoders[col] = le
        cat_cardinalities.append(len(le.classes_))
    X_train_cat = np.column_stack(X_train_cat) if cat_cols else np.empty((len(X_train), 0))
    X_val_cat = np.column_stack(X_val_cat) if cat_cols else np.empty((len(X_val), 0))
    X_test_cat = np.column_stack(X_test_cat) if cat_cols else np.empty((len(X_test), 0))
    return (X_train_num, X_train_cat), (X_val_num, X_val_cat), (X_test_num, X_test_cat), scaler, cat_cardinalities
'''

with open(os.path.join(project_root, 'src', 'preprocess.py'), 'w', encoding='utf-8') as f:
    f.write(preprocess_code)
print("Created src/preprocess.py")

# 2. train.py (包含所有模型的训练和评估函数)
train_code = '''import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import joblib
from src.models.ft_transformer import FTTransformer
from src.utils import set_seed

def train_xgboost(X_train, y_train, X_val, y_val, params, seed=42):
    model = XGBClassifier(random_state=seed, eval_metric='logloss', **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model

def train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    model = LGBMClassifier(random_state=seed, **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[], verbose=False)
    return model

def train_logistic(X_train, y_train, X_val, y_val, params, seed=42):
    model = LogisticRegression(random_state=seed, max_iter=1000, **params)
    model.fit(X_train, y_train)
    return model

def train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    model = TabNetClassifier(seed=seed, **params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=['auc'],
        max_epochs=50,
        patience=10,
        batch_size=256,
        virtual_batch_size=128,
        verbose=0
    )
    return model

def train_ft_transformer(X_num_train, X_cat_train, y_train,
                         X_num_val, X_cat_val, y_val,
                         cat_cardinalities, params, seed=42):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FTTransformer(
        n_num_features=X_num_train.shape[1],
        cat_cardinalities=cat_cardinalities,
        embedding_dim=params.get('embedding_dim', 64),
        n_blocks=params.get('n_blocks', 3),
        n_heads=params.get('n_heads', 8),
        ff_dim=params.get('ff_dim', 128),
        dropout=params.get('dropout', 0.1),
        output_dim=1
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=params.get('lr', 1e-3))
    criterion = nn.BCEWithLogitsLoss()
    
    train_dataset = TensorDataset(torch.FloatTensor(X_num_train), torch.LongTensor(X_cat_train), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_num_val), torch.LongTensor(X_cat_val), torch.FloatTensor(y_val))
    train_loader = DataLoader(train_dataset, batch_size=params.get('batch_size', 256), shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256)
    
    best_val_auc = 0
    best_model_state = None
    patience = 10
    epochs_no_improve = 0
    for epoch in range(params.get('epochs', 100)):
        model.train()
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x_num, x_cat)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        model.eval()
        y_pred_list = []
        y_true_list = []
        with torch.no_grad():
            for x_num, x_cat, y in val_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_list.append(probs.cpu().numpy())
                y_true_list.append(y.cpu().numpy())
        y_pred = np.concatenate(y_pred_list)
        y_true = np.concatenate(y_true_list)
        val_auc = roc_auc_score(y_true, y_pred)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            break
    model.load_state_dict(best_model_state)
    return model

def evaluate_model(model, X_test, y_test, model_type, cat_cardinalities=None, X_cat_test=None):
    # Inference latency and metrics
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        def predict_proba_batch(X):
            if model_type == 'logistic':
                return model.predict_proba(X)[:, 1]
            else:
                return model.predict_proba(X)[:, 1]
        sample_X = X_test[:1000]
        start = time.perf_counter()
        for _ in range(10):
            _ = predict_proba_batch(sample_X)
        latency = (time.perf_counter() - start) / (10 * len(sample_X)) * 1000
        y_pred_proba = predict_proba_batch(X_test)
    elif model_type == 'tabnet':
        start = time.perf_counter()
        for _ in range(10):
            _ = model.predict_proba(X_test[:1000])[:, 1]
        latency = (time.perf_counter() - start) / (10 * 1000) * 1000
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif model_type == 'ft_transformer':
        device = next(model.parameters()).device
        model.eval()
        X_num_test, X_cat_test = X_test
        test_dataset = TensorDataset(torch.FloatTensor(X_num_test), torch.LongTensor(X_cat_test))
        test_loader = DataLoader(test_dataset, batch_size=256)
        y_pred_proba_list = []
        start = time.perf_counter()
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_proba_list.append(probs.cpu().numpy())
        latency = (time.perf_counter() - start) / len(X_num_test) * 1000
        y_pred_proba = np.concatenate(y_pred_proba_list)
    
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    # Model size
    import tempfile
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
            joblib.dump(model, f.name)
            size_mb = os.path.getsize(f.name) / (1024 * 1024)
            os.unlink(f.name)
    elif model_type == 'tabnet':
        with tempfile.NamedTemporaryFile(suffix='.zip', delete=False) as f:
            model.save_model(f.name)
            size_mb = os.path.getsize(f.name) / (1024 * 1024)
            os.unlink(f.name)
    else:
        with tempfile.NamedTemporaryFile(suffix='.pt', delete=False) as f:
            torch.save(model.state_dict(), f.name)
            size_mb = os.path.getsize(f.name) / (1024 * 1024)
            os.unlink(f.name)
    
    return {'auc': auc, 'accuracy': acc, 'f1': f1, 'latency_ms_per_sample': latency, 'model_size_mb': size_mb}
'''

with open(os.path.join(project_root, 'src', 'train.py'), 'w', encoding='utf-8') as f:
    f.write(train_code)
print("Created src/train.py")

# 3. tune.py
tune_code = '''import optuna
from optuna.samplers import TPESampler
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from src.train import train_xgboost, train_lightgbm, train_logistic, train_tabnet, train_ft_transformer

def objective_xgboost(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    model = train_xgboost(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_lightgbm(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    model = train_lightgbm(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_logistic(trial, X_train, y_train, X_val, y_val):
    params = {
        'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        'penalty': 'l2',
        'solver': 'lbfgs'
    }
    model = train_logistic(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_tabnet(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_d': trial.suggest_int('n_d', 8, 64),
        'n_a': trial.suggest_int('n_a', 8, 64),
        'n_steps': trial.suggest_int('n_steps', 3, 10),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-5, 1e-3, log=True),
    }
    model = train_tabnet(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_ft_transformer(trial, X_num_train, X_cat_train, y_train, X_num_val, X_cat_val, y_val, cat_cardinalities):
    params = {
        'embedding_dim': trial.suggest_int('embedding_dim', 32, 128),
        'n_blocks': trial.suggest_int('n_blocks', 2, 6),
        'n_heads': trial.suggest_int('n_heads', 4, 16),
        'ff_dim': trial.suggest_int('ff_dim', 64, 256),
        'dropout': trial.suggest_float('dropout', 0.0, 0.3),
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [128, 256, 512]),
        'epochs': 50,
    }
    model = train_ft_transformer(X_num_train, X_cat_train, y_train,
                                 X_num_val, X_cat_val, y_val,
                                 cat_cardinalities, params)
    device = next(model.parameters()).device
    model.eval()
    y_pred_list = []
    with torch.no_grad():
        from torch.utils.data import DataLoader, TensorDataset
        val_dataset = TensorDataset(torch.FloatTensor(X_num_val), torch.LongTensor(X_cat_val))
        val_loader = DataLoader(val_dataset, batch_size=256)
        for x_num, x_cat in val_loader:
            x_num, x_cat = x_num.to(device), x_cat.to(device)
            logits = model(x_num, x_cat)
            probs = torch.sigmoid(logits)
            y_pred_list.append(probs.cpu().numpy())
    y_pred = np.concatenate(y_pred_list)
    return roc_auc_score(y_val, y_pred)

def tune_model(model_name, train_data, val_data, cat_cardinalities=None, n_trials=30):
    if model_name == 'xgboost':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_xgboost(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'lightgbm':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_lightgbm(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'logistic':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_logistic(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'tabnet':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_tabnet(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'ft_transformer':
        (X_num_train, X_cat_train), y_train = train_data
        (X_num_val, X_cat_val), y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_ft_transformer(t, X_num_train, X_cat_train, y_train,
                                                          X_num_val, X_cat_val, y_val,
                                                          cat_cardinalities), n_trials=n_trials)
        return study.best_params
    else:
        raise ValueError(f"Unknown model {model_name}")
'''

with open(os.path.join(project_root, 'src', 'tune.py'), 'w', encoding='utf-8') as f:
    f.write(tune_code)
print("Created src/tune.py")

# 4. run_experiments.py
run_code = '''import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import pandas as pd
import numpy as np
import joblib
import torch
from src.preprocess import *
from src.train import *
from src.tune import tune_model
from src.utils import set_seed, save_results

def run_experiment(dataset_name='adult', data_path='data/raw/adult.data', seeds=[42, 123, 2024]):
    X, y = load_adult_data(data_path)
    num_cols, cat_cols = get_feature_types(X)
    
    all_results = []
    for seed in seeds:
        print(f"\\n========== Seed {seed} ==========")
        set_seed(seed)
        (X_train, y_train), (X_val, y_val), (X_test, y_test) = split_data(X, y, random_state=seed)
        
        # Tree preprocessing
        X_train_tree, X_val_tree, X_test_tree, _ = preprocess_for_tree(X_train, X_val, X_test, cat_cols)
        # Linear preprocessing
        X_train_lr, X_val_lr, X_test_lr, _, _ = preprocess_for_linear(X_train, X_val, X_test, num_cols, cat_cols)
        # DL preprocessing
        (X_num_train, X_cat_train), (X_num_val, X_cat_val), (X_num_test, X_cat_test), _, cat_cardinalities = preprocess_for_dl(
            X_train, X_val, X_test, num_cols, cat_cols)
        
        models = ['xgboost', 'lightgbm', 'logistic', 'tabnet', 'ft_transformer']
        for model_name in models:
            print(f"  Running {model_name}...")
            if model_name in ['xgboost', 'lightgbm', 'logistic', 'tabnet']:
                best_params = tune_model(model_name, (X_train_tree, y_train), (X_val_tree, y_val), n_trials=10)
                if model_name == 'xgboost':
                    model = train_xgboost(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
                elif model_name == 'lightgbm':
                    model = train_lightgbm(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
                elif model_name == 'logistic':
                    model = train_logistic(X_train_lr, y_train, X_val_lr, y_val, best_params, seed)
                    X_test_model = X_test_lr
                elif model_name == 'tabnet':
                    model = train_tabnet(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
            else:  # ft_transformer
                best_params = tune_model(model_name, ((X_num_train, X_cat_train), y_train),
                                         ((X_num_val, X_cat_val), y_val),
                                         cat_cardinalities=cat_cardinalities, n_trials=10)
                model = train_ft_transformer(X_num_train, X_cat_train, y_train,
                                             X_num_val, X_cat_val, y_val,
                                             cat_cardinalities, best_params, seed)
                X_test_model = (X_num_test, X_cat_test)
            
            eval_metrics = evaluate_model(model, X_test_model, y_test, model_name, cat_cardinalities, X_cat_test if model_name=='ft_transformer' else None)
            eval_metrics['model'] = model_name
            eval_metrics['seed'] = seed
            eval_metrics['best_params'] = best_params
            all_results.append(eval_metrics)
            
            os.makedirs(f'results/models/{dataset_name}/seed{seed}', exist_ok=True)
            if model_name == 'tabnet':
                model.save_model(f'results/models/{dataset_name}/seed{seed}/{model_name}.zip')
            elif model_name == 'ft_transformer':
                torch.save(model.state_dict(), f'results/models/{dataset_name}/seed{seed}/{model_name}.pt')
            else:
                joblib.dump(model, f'results/models/{dataset_name}/seed{seed}/{model_name}.pkl')
    
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(f'results/{dataset_name}_results.csv', index=False)
    print("\\nAll experiments completed. Results saved to results/")
    return results_df

if __name__ == '__main__':
    run_experiment()
'''

with open(os.path.join(project_root, 'src', 'run_experiments.py'), 'w', encoding='utf-8') as f:
    f.write(run_code)
print("Created src/run_experiments.py")

print("\\n所有文件创建完成！现在请确保已下载 Adult 数据集到 data/raw/adult.data")

Created src/preprocess.py
Created src/train.py
Created src/tune.py
Created src/run_experiments.py
\n所有文件创建完成！现在请确保已下载 Adult 数据集到 data/raw/adult.data


In [14]:
import os
import urllib.request

# 确保目录存在
os.makedirs('data/raw', exist_ok=True)

# 下载 Adult 数据集
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
urllib.request.urlretrieve(url, 'data/raw/adult.data')
print("数据集下载完成，保存在 data/raw/adult.data")

数据集下载完成，保存在 data/raw/adult.data


In [15]:
import os
import urllib.request

# 创建目录
os.makedirs('data/raw', exist_ok=True)

# 下载数据集
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
output_path = 'data/raw/adult.data'

if not os.path.exists(output_path):
    urllib.request.urlretrieve(url, output_path)
    print("数据集已下载到", output_path)
else:
    print("数据集已存在")

数据集已存在


In [16]:
import os
import tempfile
import joblib
import torch
from src.train import evaluate_model  # 先导入原有函数（但我们会覆盖）

# 重新定义 evaluate_model 函数，避免 Windows 权限错误
fixed_evaluate_code = '''
def evaluate_model(model, X_test, y_test, model_type, cat_cardinalities=None, X_cat_test=None):
    import time
    import numpy as np
    import torch
    from torch.utils.data import DataLoader, TensorDataset
    from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
    import os
    import tempfile
    import joblib

    # 推理延迟和预测
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        def predict_proba_batch(X):
            if model_type == 'logistic':
                return model.predict_proba(X)[:, 1]
            else:
                return model.predict_proba(X)[:, 1]
        sample_X = X_test[:1000]
        start = time.perf_counter()
        for _ in range(10):
            _ = predict_proba_batch(sample_X)
        latency = (time.perf_counter() - start) / (10 * len(sample_X)) * 1000
        y_pred_proba = predict_proba_batch(X_test)
    elif model_type == 'tabnet':
        start = time.perf_counter()
        for _ in range(10):
            _ = model.predict_proba(X_test[:1000])[:, 1]
        latency = (time.perf_counter() - start) / (10 * 1000) * 1000
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif model_type == 'ft_transformer':
        device = next(model.parameters()).device
        model.eval()
        X_num_test, X_cat_test = X_test
        test_dataset = TensorDataset(torch.FloatTensor(X_num_test), torch.LongTensor(X_cat_test))
        test_loader = DataLoader(test_dataset, batch_size=256)
        y_pred_proba_list = []
        start = time.perf_counter()
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_proba_list.append(probs.cpu().numpy())
        latency = (time.perf_counter() - start) / len(X_num_test) * 1000
        y_pred_proba = np.concatenate(y_pred_proba_list)
    
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    # 模型大小（使用临时目录而非 NamedTemporaryFile，避免权限问题）
    import tempfile
    tmp_dir = tempfile.mkdtemp()  # 创建临时目录
    try:
        if model_type in ['xgboost', 'lightgbm', 'logistic']:
            tmp_path = os.path.join(tmp_dir, 'model.pkl')
            joblib.dump(model, tmp_path)
            size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
        elif model_type == 'tabnet':
            tmp_path = os.path.join(tmp_dir, 'model.zip')
            model.save_model(tmp_path)
            size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
        else:
            tmp_path = os.path.join(tmp_dir, 'model.pt')
            torch.save(model.state_dict(), tmp_path)
            size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    finally:
        # 清理临时文件
        import shutil
        shutil.rmtree(tmp_dir, ignore_errors=True)
    
    return {'auc': auc, 'accuracy': acc, 'f1': f1, 'latency_ms_per_sample': latency, 'model_size_mb': size_mb}
'''

# 读取当前的 train.py 文件，替换 evaluate_model 函数
train_path = os.path.join('src', 'train.py')
with open(train_path, 'r', encoding='utf-8') as f:
    content = f.read()

# 找到 evaluate_model 函数开始和结束的位置（简单方式：替换整个文件内容，但保留其他函数）
# 更安全的方法：用正则或字符串替换。由于我们已经知道函数定义，可以直接将新的 evaluate_model 代码插入。
# 为了简单，我们重新写入整个 train.py 文件，但保留前面的代码。
# 先读取原文件，找到 "def evaluate_model" 之前的代码和之后的代码，然后替换。
import re
pattern = r'(.*?)def evaluate_model\(.*?\):.*?(?=\n\ndef |\Z)'
# 这个正则比较复杂，建议直接手动替换：我们将原文件中的 evaluate_model 函数替换为新版本。
# 更简单的方法：直接在 notebook 中运行修复后的训练流程，而不修改原文件（临时覆盖）。
# 但为了持久化，我提供修改后的完整 train.py 文件内容。由于内容很长，我们只替换函数部分。

# 使用正则替换
new_content = re.sub(
    r'def evaluate_model\(.*?\):.*?(?=\n\ndef |\Z)',
    fixed_evaluate_code.strip(),
    content,
    flags=re.DOTALL
)

# 备份原文件
import shutil
shutil.copy(train_path, train_path + '.bak')
# 写入新内容
with open(train_path, 'w', encoding='utf-8') as f:
    f.write(new_content)
print("train.py 已更新，修复了临时文件权限问题")

train.py 已更新，修复了临时文件权限问题


In [17]:
import os

train_py_path = r'C:\Users\king\Desktop\tabular_dl_project\src\train.py'

fixed_train_code = '''import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import joblib
from src.models.ft_transformer import FTTransformer
from src.utils import set_seed

def train_xgboost(X_train, y_train, X_val, y_val, params, seed=42):
    model = XGBClassifier(random_state=seed, eval_metric='logloss', **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model

def train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    model = LGBMClassifier(random_state=seed, **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[], verbose=False)
    return model

def train_logistic(X_train, y_train, X_val, y_val, params, seed=42):
    model = LogisticRegression(random_state=seed, max_iter=1000, **params)
    model.fit(X_train, y_train)
    return model

def train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    model = TabNetClassifier(seed=seed, **params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=['auc'],
        max_epochs=50,
        patience=10,
        batch_size=256,
        virtual_batch_size=128,
        verbose=0
    )
    return model

def train_ft_transformer(X_num_train, X_cat_train, y_train,
                         X_num_val, X_cat_val, y_val,
                         cat_cardinalities, params, seed=42):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FTTransformer(
        n_num_features=X_num_train.shape[1],
        cat_cardinalities=cat_cardinalities,
        embedding_dim=params.get('embedding_dim', 64),
        n_blocks=params.get('n_blocks', 3),
        n_heads=params.get('n_heads', 8),
        ff_dim=params.get('ff_dim', 128),
        dropout=params.get('dropout', 0.1),
        output_dim=1
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=params.get('lr', 1e-3))
    criterion = nn.BCEWithLogitsLoss()
    
    train_dataset = TensorDataset(torch.FloatTensor(X_num_train), torch.LongTensor(X_cat_train), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_num_val), torch.LongTensor(X_cat_val), torch.FloatTensor(y_val))
    train_loader = DataLoader(train_dataset, batch_size=params.get('batch_size', 256), shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256)
    
    best_val_auc = 0
    best_model_state = None
    patience = 10
    epochs_no_improve = 0
    for epoch in range(params.get('epochs', 100)):
        model.train()
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x_num, x_cat)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        model.eval()
        y_pred_list = []
        y_true_list = []
        with torch.no_grad():
            for x_num, x_cat, y in val_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_list.append(probs.cpu().numpy())
                y_true_list.append(y.cpu().numpy())
        y_pred = np.concatenate(y_pred_list)
        y_true = np.concatenate(y_true_list)
        val_auc = roc_auc_score(y_true, y_pred)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            break
    model.load_state_dict(best_model_state)
    return model

def evaluate_model(model, X_test, y_test, model_type, cat_cardinalities=None, X_cat_test=None):
    # 修复版本：使用临时目录而不是 NamedTemporaryFile
    import tempfile
    import shutil
    
    # 推理延迟和预测
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        def predict_proba_batch(X):
            if model_type == 'logistic':
                return model.predict_proba(X)[:, 1]
            else:
                return model.predict_proba(X)[:, 1]
        sample_X = X_test[:1000]
        start = time.perf_counter()
        for _ in range(10):
            _ = predict_proba_batch(sample_X)
        latency = (time.perf_counter() - start) / (10 * len(sample_X)) * 1000
        y_pred_proba = predict_proba_batch(X_test)
    elif model_type == 'tabnet':
        start = time.perf_counter()
        for _ in range(10):
            _ = model.predict_proba(X_test[:1000])[:, 1]
        latency = (time.perf_counter() - start) / (10 * 1000) * 1000
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif model_type == 'ft_transformer':
        device = next(model.parameters()).device
        model.eval()
        X_num_test, X_cat_test = X_test
        test_dataset = TensorDataset(torch.FloatTensor(X_num_test), torch.LongTensor(X_cat_test))
        test_loader = DataLoader(test_dataset, batch_size=256)
        y_pred_proba_list = []
        start = time.perf_counter()
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_proba_list.append(probs.cpu().numpy())
        latency = (time.perf_counter() - start) / len(X_num_test) * 1000
        y_pred_proba = np.concatenate(y_pred_proba_list)
    
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    # 模型大小：使用临时目录，避免 Windows 文件锁定问题
    tmp_dir = tempfile.mkdtemp()
    try:
        if model_type in ['xgboost', 'lightgbm', 'logistic']:
            tmp_path = os.path.join(tmp_dir, 'model.pkl')
            joblib.dump(model, tmp_path)
        elif model_type == 'tabnet':
            tmp_path = os.path.join(tmp_dir, 'model.zip')
            model.save_model(tmp_path)
        else:
            tmp_path = os.path.join(tmp_dir, 'model.pt')
            torch.save(model.state_dict(), tmp_path)
        size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    
    return {'auc': auc, 'accuracy': acc, 'f1': f1, 'latency_ms_per_sample': latency, 'model_size_mb': size_mb}
'''

with open(train_py_path, 'w', encoding='utf-8') as f:
    f.write(fixed_train_code)
print("train.py 已完全覆盖为修复版本")

train.py 已完全覆盖为修复版本


In [18]:
import time
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import tempfile
import shutil
import joblib

def safe_evaluate_model(model, X_test, y_test, model_type, cat_cardinalities=None, X_cat_test=None):
    """修复 Windows 权限问题的评估函数"""
    # 推理延迟和预测
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        def predict_proba_batch(X):
            if model_type == 'logistic':
                return model.predict_proba(X)[:, 1]
            else:
                return model.predict_proba(X)[:, 1]
        sample_X = X_test[:1000]
        start = time.perf_counter()
        for _ in range(10):
            _ = predict_proba_batch(sample_X)
        latency = (time.perf_counter() - start) / (10 * len(sample_X)) * 1000
        y_pred_proba = predict_proba_batch(X_test)
    elif model_type == 'tabnet':
        start = time.perf_counter()
        for _ in range(10):
            _ = model.predict_proba(X_test[:1000])[:, 1]
        latency = (time.perf_counter() - start) / (10 * 1000) * 1000
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif model_type == 'ft_transformer':
        device = next(model.parameters()).device
        model.eval()
        X_num_test, X_cat_test = X_test
        test_dataset = TensorDataset(torch.FloatTensor(X_num_test), torch.LongTensor(X_cat_test))
        test_loader = DataLoader(test_dataset, batch_size=256)
        y_pred_proba_list = []
        start = time.perf_counter()
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_proba_list.append(probs.cpu().numpy())
        latency = (time.perf_counter() - start) / len(X_num_test) * 1000
        y_pred_proba = np.concatenate(y_pred_proba_list)
    
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    # 模型大小：使用临时目录而非 NamedTemporaryFile
    tmp_dir = tempfile.mkdtemp()
    try:
        if model_type in ['xgboost', 'lightgbm', 'logistic']:
            tmp_path = os.path.join(tmp_dir, 'model.pkl')
            joblib.dump(model, tmp_path)
        elif model_type == 'tabnet':
            tmp_path = os.path.join(tmp_dir, 'model.zip')
            model.save_model(tmp_path)
        else:
            tmp_path = os.path.join(tmp_dir, 'model.pt')
            torch.save(model.state_dict(), tmp_path)
        size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    
    return {
        'auc': auc,
        'accuracy': acc,
        'f1': f1,
        'latency_ms_per_sample': latency,
        'model_size_mb': size_mb
    }

In [19]:
import src.train
src.train.evaluate_model = safe_evaluate_model
print("已临时替换 evaluate_model 函数，权限问题已解决")

已临时替换 evaluate_model 函数，权限问题已解决


In [20]:
import os

# 修复 train_lightgbm 函数
train_path = os.path.join('src', 'train.py')
with open(train_path, 'r', encoding='utf-8') as f:
    content = f.read()

# 替换 train_lightgbm 中的 fit 调用
old_line = "    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[], verbose=False)"
new_line = "    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[], verbose=0)"
if old_line in content:
    content = content.replace(old_line, new_line)
    with open(train_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("已修复 train_lightgbm 的 verbose 参数")
else:
    print("未找到对应行，尝试其他替换")
    # 备用替换：去掉 verbose 参数
    alt_line = "    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[], verbose=False)"
    new_alt = "    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], callbacks=[])"
    content = content.replace(alt_line, new_alt)
    with open(train_path, 'w', encoding='utf-8') as f:
        f.write(content)
    print("已移除 verbose 参数")

已修复 train_lightgbm 的 verbose 参数


In [21]:
import src.train

# 修复 lightgbm 训练函数
def fixed_train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    from lightgbm import LGBMClassifier
    model = LGBMClassifier(random_state=seed, **params)
    # 新版 lightgbm 移除了 verbose 参数，改用 callbacks
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        callbacks=[],
        # 不传 verbose
    )
    return model

# 替换原函数
src.train.train_lightgbm = fixed_train_lightgbm
print("已修复 train_lightgbm 函数")

已修复 train_lightgbm 函数


In [22]:
import src.train

# 修复 lightgbm 训练函数（确保无 verbose）
def fixed_train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    from lightgbm import LGBMClassifier
    model = LGBMClassifier(random_state=seed, **params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric='auc',
        verbose_eval=False
    )
    return model

# 修复 tabnet 训练函数（移除 verbose）
def fixed_train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    from pytorch_tabnet.tab_model import TabNetClassifier
    model = TabNetClassifier(seed=seed, **params)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        eval_metric=['auc'],
        max_epochs=50,
        patience=10,
        batch_size=256,
        virtual_batch_size=128,
        # 注意：此处不要写 verbose 参数
    )
    return model

# 替换原函数
src.train.train_lightgbm = fixed_train_lightgbm
src.train.train_tabnet = fixed_train_tabnet
print("已修复 train_lightgbm 和 train_tabnet")

已修复 train_lightgbm 和 train_tabnet


In [23]:
import src.train

# 极简版 lightgbm 训练函数（无 eval_set, verbose 等）
def minimal_train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    from lightgbm import LGBMClassifier
    model = LGBMClassifier(random_state=seed, **params)
    model.fit(X_train, y_train)  # 只训练，不传验证集
    return model

# 极简版 tabnet 训练函数（移除所有额外参数）
def minimal_train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    from pytorch_tabnet.tab_model import TabNetClassifier
    model = TabNetClassifier(seed=seed, **params)
    model.fit(X_train, y_train)  # 只训练，不传验证集、不设 epoch 等
    return model

# 替换
src.train.train_lightgbm = minimal_train_lightgbm
src.train.train_tabnet = minimal_train_tabnet

print("已替换为极简版训练函数（无验证集反馈）")

已替换为极简版训练函数（无验证集反馈）


In [24]:
%%writefile src/train.py
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import joblib
import tempfile
import shutil
from src.models.ft_transformer import FTTransformer
from src.utils import set_seed

# ---------- 树模型 / 线性模型 ----------
def train_xgboost(X_train, y_train, X_val, y_val, params, seed=42):
    model = XGBClassifier(random_state=seed, eval_metric='logloss', **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model

def train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    model = LGBMClassifier(random_state=seed, **params)
    # 极简 fit：不传任何额外参数，避免版本兼容问题
    model.fit(X_train, y_train)
    return model

def train_logistic(X_train, y_train, X_val, y_val, params, seed=42):
    model = LogisticRegression(random_state=seed, max_iter=1000, **params)
    model.fit(X_train, y_train)
    return model

def train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    model = TabNetClassifier(seed=seed, **params)
    # 极简 fit：只传训练数据
    model.fit(X_train, y_train)
    return model

# ---------- FT-Transformer ----------
def train_ft_transformer(X_num_train, X_cat_train, y_train,
                         X_num_val, X_cat_val, y_val,
                         cat_cardinalities, params, seed=42):
    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FTTransformer(
        n_num_features=X_num_train.shape[1],
        cat_cardinalities=cat_cardinalities,
        embedding_dim=params.get('embedding_dim', 64),
        n_blocks=params.get('n_blocks', 3),
        n_heads=params.get('n_heads', 8),
        ff_dim=params.get('ff_dim', 128),
        dropout=params.get('dropout', 0.1),
        output_dim=1
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=params.get('lr', 1e-3))
    criterion = nn.BCEWithLogitsLoss()
    
    train_dataset = TensorDataset(torch.FloatTensor(X_num_train), torch.LongTensor(X_cat_train), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_num_val), torch.LongTensor(X_cat_val), torch.FloatTensor(y_val))
    train_loader = DataLoader(train_dataset, batch_size=params.get('batch_size', 256), shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256)
    
    best_val_auc = 0
    best_model_state = None
    patience = 10
    epochs_no_improve = 0
    for epoch in range(params.get('epochs', 50)):
        model.train()
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x_num, x_cat)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        model.eval()
        y_pred_list = []
        y_true_list = []
        with torch.no_grad():
            for x_num, x_cat, y in val_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_list.append(probs.cpu().numpy())
                y_true_list.append(y.cpu().numpy())
        y_pred = np.concatenate(y_pred_list)
        y_true = np.concatenate(y_true_list)
        val_auc = roc_auc_score(y_true, y_pred)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            break
    model.load_state_dict(best_model_state)
    return model

# ---------- 评估函数（无权限错误） ----------
def evaluate_model(model, X_test, y_test, model_type, cat_cardinalities=None, X_cat_test=None):
    # 预测与延迟
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        def predict_proba_batch(X):
            if model_type == 'logistic':
                return model.predict_proba(X)[:, 1]
            else:
                return model.predict_proba(X)[:, 1]
        sample_X = X_test[:1000]
        start = time.perf_counter()
        for _ in range(10):
            _ = predict_proba_batch(sample_X)
        latency = (time.perf_counter() - start) / (10 * len(sample_X)) * 1000
        y_pred_proba = predict_proba_batch(X_test)
    elif model_type == 'tabnet':
        start = time.perf_counter()
        for _ in range(10):
            _ = model.predict_proba(X_test[:1000])[:, 1]
        latency = (time.perf_counter() - start) / (10 * 1000) * 1000
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif model_type == 'ft_transformer':
        device = next(model.parameters()).device
        model.eval()
        X_num_test, X_cat_test = X_test
        test_dataset = TensorDataset(torch.FloatTensor(X_num_test), torch.LongTensor(X_cat_test))
        test_loader = DataLoader(test_dataset, batch_size=256)
        y_pred_proba_list = []
        start = time.perf_counter()
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_proba_list.append(probs.cpu().numpy())
        latency = (time.perf_counter() - start) / len(X_num_test) * 1000
        y_pred_proba = np.concatenate(y_pred_proba_list)
    
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    # 模型大小（临时目录）
    tmp_dir = tempfile.mkdtemp()
    try:
        if model_type in ['xgboost', 'lightgbm', 'logistic']:
            tmp_path = os.path.join(tmp_dir, 'model.pkl')
            joblib.dump(model, tmp_path)
        elif model_type == 'tabnet':
            tmp_path = os.path.join(tmp_dir, 'model.zip')
            model.save_model(tmp_path)
        else:
            tmp_path = os.path.join(tmp_dir, 'model.pt')
            torch.save(model.state_dict(), tmp_path)
        size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    
    return {
        'auc': auc,
        'accuracy': acc,
        'f1': f1,
        'latency_ms_per_sample': latency,
        'model_size_mb': size_mb
    }

Overwriting src/train.py


In [25]:
%%writefile src/tune.py
import optuna
from optuna.samplers import TPESampler
import numpy as np
import torch
from sklearn.metrics import roc_auc_score
from src.train import train_xgboost, train_lightgbm, train_logistic, train_tabnet, train_ft_transformer

def objective_xgboost(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    model = train_xgboost(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_lightgbm(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
    }
    model = train_lightgbm(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_logistic(trial, X_train, y_train, X_val, y_val):
    params = {
        'C': trial.suggest_float('C', 1e-4, 1e2, log=True),
        'penalty': 'l2',
        'solver': 'lbfgs'
    }
    model = train_logistic(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_tabnet(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_d': trial.suggest_int('n_d', 8, 64),
        'n_a': trial.suggest_int('n_a', 8, 64),
        'n_steps': trial.suggest_int('n_steps', 3, 10),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-5, 1e-3, log=True),
    }
    model = train_tabnet(X_train, y_train, X_val, y_val, params)
    y_pred = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred)

def objective_ft_transformer(trial, X_num_train, X_cat_train, y_train, X_num_val, X_cat_val, y_val, cat_cardinalities):
    params = {
        'embedding_dim': trial.suggest_int('embedding_dim', 32, 128),
        'n_blocks': trial.suggest_int('n_blocks', 2, 6),
        'n_heads': trial.suggest_int('n_heads', 4, 16),
        'ff_dim': trial.suggest_int('ff_dim', 64, 256),
        'dropout': trial.suggest_float('dropout', 0.0, 0.3),
        'lr': trial.suggest_float('lr', 1e-4, 1e-2, log=True),
        'batch_size': trial.suggest_categorical('batch_size', [128, 256, 512]),
        'epochs': 50,
    }
    model = train_ft_transformer(X_num_train, X_cat_train, y_train,
                                 X_num_val, X_cat_val, y_val,
                                 cat_cardinalities, params)
    device = next(model.parameters()).device
    model.eval()
    y_pred_list = []
    with torch.no_grad():
        from torch.utils.data import DataLoader, TensorDataset
        val_dataset = TensorDataset(torch.FloatTensor(X_num_val), torch.LongTensor(X_cat_val))
        val_loader = DataLoader(val_dataset, batch_size=256)
        for x_num, x_cat in val_loader:
            x_num, x_cat = x_num.to(device), x_cat.to(device)
            logits = model(x_num, x_cat)
            probs = torch.sigmoid(logits)
            y_pred_list.append(probs.cpu().numpy())
    y_pred = np.concatenate(y_pred_list)
    return roc_auc_score(y_val, y_pred)

def tune_model(model_name, train_data, val_data, cat_cardinalities=None, n_trials=30):
    if model_name == 'xgboost':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_xgboost(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'lightgbm':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_lightgbm(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'logistic':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_logistic(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'tabnet':
        X_train, y_train = train_data
        X_val, y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_tabnet(t, X_train, y_train, X_val, y_val), n_trials=n_trials)
        return study.best_params
    elif model_name == 'ft_transformer':
        (X_num_train, X_cat_train), y_train = train_data
        (X_num_val, X_cat_val), y_val = val_data
        study = optuna.create_study(direction='maximize', sampler=TPESampler(seed=42))
        study.optimize(lambda t: objective_ft_transformer(t, X_num_train, X_cat_train, y_train,
                                                          X_num_val, X_cat_val, y_val,
                                                          cat_cardinalities), n_trials=n_trials)
        return study.best_params
    else:
        raise ValueError(f"Unknown model {model_name}")

Overwriting src/tune.py


In [26]:
%%writefile src/run_experiments.py
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import pandas as pd
import numpy as np
import joblib
import torch
from src.preprocess import *
from src.train import *
from src.tune import tune_model
from src.utils import set_seed, save_results

def run_experiment(dataset_name='adult', data_path='data/raw/adult.data', seeds=[42, 123, 2024]):
    X, y = load_adult_data(data_path)
    num_cols, cat_cols = get_feature_types(X)
    
    all_results = []
    for seed in seeds:
        print(f"\n========== Seed {seed} ==========")
        set_seed(seed)
        (X_train, y_train), (X_val, y_val), (X_test, y_test) = split_data(X, y, random_state=seed)
        
        # Tree preprocessing
        X_train_tree, X_val_tree, X_test_tree, _ = preprocess_for_tree(X_train, X_val, X_test, cat_cols)
        # Linear preprocessing
        X_train_lr, X_val_lr, X_test_lr, _, _ = preprocess_for_linear(X_train, X_val, X_test, num_cols, cat_cols)
        # DL preprocessing
        (X_num_train, X_cat_train), (X_num_val, X_cat_val), (X_num_test, X_cat_test), _, cat_cardinalities = preprocess_for_dl(
            X_train, X_val, X_test, num_cols, cat_cols)
        
        models = ['xgboost', 'lightgbm', 'logistic', 'tabnet', 'ft_transformer']
        for model_name in models:
            print(f"  Running {model_name}...")
            if model_name in ['xgboost', 'lightgbm', 'logistic', 'tabnet']:
                # 调参次数改为 2 以便快速测试（正式运行时改为 30）
                best_params = tune_model(model_name, (X_train_tree, y_train), (X_val_tree, y_val), n_trials=2)
                if model_name == 'xgboost':
                    model = train_xgboost(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
                elif model_name == 'lightgbm':
                    model = train_lightgbm(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
                elif model_name == 'logistic':
                    model = train_logistic(X_train_lr, y_train, X_val_lr, y_val, best_params, seed)
                    X_test_model = X_test_lr
                elif model_name == 'tabnet':
                    model = train_tabnet(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
            else:  # ft_transformer
                best_params = tune_model(model_name, ((X_num_train, X_cat_train), y_train),
                                         ((X_num_val, X_cat_val), y_val),
                                         cat_cardinalities=cat_cardinalities, n_trials=2)
                model = train_ft_transformer(X_num_train, X_cat_train, y_train,
                                             X_num_val, X_cat_val, y_val,
                                             cat_cardinalities, best_params, seed)
                X_test_model = (X_num_test, X_cat_test)
            
            eval_metrics = evaluate_model(model, X_test_model, y_test, model_name, cat_cardinalities, X_cat_test if model_name=='ft_transformer' else None)
            eval_metrics['model'] = model_name
            eval_metrics['seed'] = seed
            eval_metrics['best_params'] = best_params
            all_results.append(eval_metrics)
            
            os.makedirs(f'results/models/{dataset_name}/seed{seed}', exist_ok=True)
            if model_name == 'tabnet':
                model.save_model(f'results/models/{dataset_name}/seed{seed}/{model_name}.zip')
            elif model_name == 'ft_transformer':
                torch.save(model.state_dict(), f'results/models/{dataset_name}/seed{seed}/{model_name}.pt')
            else:
                joblib.dump(model, f'results/models/{dataset_name}/seed{seed}/{model_name}.pkl')
    
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(f'results/{dataset_name}_results.csv', index=False)
    print("\nAll experiments completed. Results saved to results/")
    return results_df

if __name__ == '__main__':
    run_experiment()

Overwriting src/run_experiments.py


In [27]:
import src.train
import numpy as np

# 修复 train_tabnet：将 DataFrame 转为 numpy 数组
def fixed_train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    from pytorch_tabnet.tab_model import TabNetClassifier
    # 转换为 numpy 数组（TabNet 不支持 DataFrame）
    if hasattr(X_train, 'values'):
        X_train = X_train.values
        X_val = X_val.values
    if hasattr(y_train, 'values'):
        y_train = y_train.values
        y_val = y_val.values
    model = TabNetClassifier(seed=seed, **params)
    model.fit(X_train, y_train)  # 只训练，不传验证集
    return model

# 替换原函数
src.train.train_tabnet = fixed_train_tabnet
print("已修复 train_tabnet：支持 DataFrame 转 numpy")

已修复 train_tabnet：支持 DataFrame 转 numpy


In [28]:
import src.tune

# 修复 objective_tabnet，将 X_val 转为 numpy 数组
def fixed_objective_tabnet(trial, X_train, y_train, X_val, y_val):
    params = {
        'n_d': trial.suggest_int('n_d', 8, 64),
        'n_a': trial.suggest_int('n_a', 8, 64),
        'n_steps': trial.suggest_int('n_steps', 3, 10),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-5, 1e-3, log=True),
    }
    # 确保传入 numpy 数组
    X_train_np = X_train.values if hasattr(X_train, 'values') else X_train
    X_val_np = X_val.values if hasattr(X_val, 'values') else X_val
    model = src.train.train_tabnet(X_train_np, y_train, X_val_np, y_val, params)
    y_pred = model.predict_proba(X_val_np)[:, 1]
    return src.tune.roc_auc_score(y_val, y_pred)

# 替换原函数
src.tune.objective_tabnet = fixed_objective_tabnet
print("已修复 objective_tabnet 中的 DataFrame 问题")

已修复 objective_tabnet 中的 DataFrame 问题


In [29]:
import src.train

def safe_train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    from pytorch_tabnet.tab_model import TabNetClassifier
    # 转换为 numpy
    X_train_np = X_train.values if hasattr(X_train, 'values') else X_train
    X_val_np = X_val.values if hasattr(X_val, 'values') else X_val
    model = TabNetClassifier(seed=seed, **params)
    model.fit(X_train_np, y_train)
    return model

src.train.train_tabnet = safe_train_tabnet
print("已修复 train_tabnet 自动转换 DataFrame")

已修复 train_tabnet 自动转换 DataFrame


In [30]:
%%writefile src/train.py
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import joblib
import tempfile
import shutil
from src.models.ft_transformer import FTTransformer
from src.utils import set_seed

# ---------- 辅助函数：确保 numpy 数组 ----------
def to_numpy(X):
    if hasattr(X, 'values'):
        return X.values
    return X

# ---------- 树模型 / 线性模型 ----------
def train_xgboost(X_train, y_train, X_val, y_val, params, seed=42):
    X_train = to_numpy(X_train)
    y_train = to_numpy(y_train)
    model = XGBClassifier(random_state=seed, eval_metric='logloss', **params)
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
    return model

def train_lightgbm(X_train, y_train, X_val, y_val, params, seed=42):
    X_train = to_numpy(X_train)
    y_train = to_numpy(y_train)
    model = LGBMClassifier(random_state=seed, **params)
    model.fit(X_train, y_train)
    return model

def train_logistic(X_train, y_train, X_val, y_val, params, seed=42):
    X_train = to_numpy(X_train)
    y_train = to_numpy(y_train)
    model = LogisticRegression(random_state=seed, max_iter=1000, **params)
    model.fit(X_train, y_train)
    return model

def train_tabnet(X_train, y_train, X_val, y_val, params, seed=42):
    # 转换为 numpy 数组
    X_train = to_numpy(X_train)
    y_train = to_numpy(y_train)
    model = TabNetClassifier(seed=seed, **params)
    model.fit(X_train, y_train)
    return model

# ---------- FT-Transformer ----------
def train_ft_transformer(X_num_train, X_cat_train, y_train,
                         X_num_val, X_cat_val, y_val,
                         cat_cardinalities, params, seed=42):
    # 确保是 numpy 数组
    X_num_train = to_numpy(X_num_train)
    X_cat_train = to_numpy(X_cat_train)
    y_train = to_numpy(y_train)
    X_num_val = to_numpy(X_num_val)
    X_cat_val = to_numpy(X_cat_val)
    y_val = to_numpy(y_val)

    set_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = FTTransformer(
        n_num_features=X_num_train.shape[1],
        cat_cardinalities=cat_cardinalities,
        embedding_dim=params.get('embedding_dim', 64),
        n_blocks=params.get('n_blocks', 3),
        n_heads=params.get('n_heads', 8),
        ff_dim=params.get('ff_dim', 128),
        dropout=params.get('dropout', 0.1),
        output_dim=1
    ).to(device)
    optimizer = optim.Adam(model.parameters(), lr=params.get('lr', 1e-3))
    criterion = nn.BCEWithLogitsLoss()
    
    train_dataset = TensorDataset(torch.FloatTensor(X_num_train), torch.LongTensor(X_cat_train), torch.FloatTensor(y_train))
    val_dataset = TensorDataset(torch.FloatTensor(X_num_val), torch.LongTensor(X_cat_val), torch.FloatTensor(y_val))
    train_loader = DataLoader(train_dataset, batch_size=params.get('batch_size', 256), shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=256)
    
    best_val_auc = 0
    best_model_state = None
    patience = 10
    epochs_no_improve = 0
    for epoch in range(params.get('epochs', 50)):
        model.train()
        for x_num, x_cat, y in train_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x_num, x_cat)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
        model.eval()
        y_pred_list = []
        y_true_list = []
        with torch.no_grad():
            for x_num, x_cat, y in val_loader:
                x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_list.append(probs.cpu().numpy())
                y_true_list.append(y.cpu().numpy())
        y_pred = np.concatenate(y_pred_list)
        y_true = np.concatenate(y_true_list)
        val_auc = roc_auc_score(y_true, y_pred)
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_model_state = {k: v.cpu() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
        if epochs_no_improve >= patience:
            break
    model.load_state_dict(best_model_state)
    return model

# ---------- 评估函数（修复 TabNet 的 DataFrame 问题） ----------
def evaluate_model(model, X_test, y_test, model_type, cat_cardinalities=None, X_cat_test=None):
    # 确保 y_test 是 numpy 数组
    y_test = to_numpy(y_test)
    
    # 预测与延迟
    if model_type in ['xgboost', 'lightgbm', 'logistic']:
        X_test = to_numpy(X_test)
        def predict_proba_batch(X):
            if model_type == 'logistic':
                return model.predict_proba(X)[:, 1]
            else:
                return model.predict_proba(X)[:, 1]
        sample_X = X_test[:1000]
        start = time.perf_counter()
        for _ in range(10):
            _ = predict_proba_batch(sample_X)
        latency = (time.perf_counter() - start) / (10 * len(sample_X)) * 1000
        y_pred_proba = predict_proba_batch(X_test)
    elif model_type == 'tabnet':
        # TabNet 需要 numpy 数组
        X_test = to_numpy(X_test)
        start = time.perf_counter()
        for _ in range(10):
            _ = model.predict_proba(X_test[:1000])[:, 1]
        latency = (time.perf_counter() - start) / (10 * 1000) * 1000
        y_pred_proba = model.predict_proba(X_test)[:, 1]
    elif model_type == 'ft_transformer':
        device = next(model.parameters()).device
        model.eval()
        X_num_test, X_cat_test = X_test
        X_num_test = to_numpy(X_num_test)
        X_cat_test = to_numpy(X_cat_test)
        test_dataset = TensorDataset(torch.FloatTensor(X_num_test), torch.LongTensor(X_cat_test))
        test_loader = DataLoader(test_dataset, batch_size=256)
        y_pred_proba_list = []
        start = time.perf_counter()
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                logits = model(x_num, x_cat)
                probs = torch.sigmoid(logits)
                y_pred_proba_list.append(probs.cpu().numpy())
        latency = (time.perf_counter() - start) / len(X_num_test) * 1000
        y_pred_proba = np.concatenate(y_pred_proba_list)
    
    y_pred_binary = (y_pred_proba >= 0.5).astype(int)
    auc = roc_auc_score(y_test, y_pred_proba)
    acc = accuracy_score(y_test, y_pred_binary)
    f1 = f1_score(y_test, y_pred_binary)
    
    # 模型大小（临时目录）
    tmp_dir = tempfile.mkdtemp()
    try:
        if model_type in ['xgboost', 'lightgbm', 'logistic']:
            tmp_path = os.path.join(tmp_dir, 'model.pkl')
            joblib.dump(model, tmp_path)
        elif model_type == 'tabnet':
            tmp_path = os.path.join(tmp_dir, 'model.zip')
            model.save_model(tmp_path)
        else:
            tmp_path = os.path.join(tmp_dir, 'model.pt')
            torch.save(model.state_dict(), tmp_path)
        size_mb = os.path.getsize(tmp_path) / (1024 * 1024)
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)
    
    return {
        'auc': auc,
        'accuracy': acc,
        'f1': f1,
        'latency_ms_per_sample': latency,
        'model_size_mb': size_mb
    }

Overwriting src/train.py


In [31]:
%%writefile src/run_experiments.py
import sys
import os
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
import pandas as pd
import numpy as np
import joblib
import torch
from src.preprocess import *
from src.train import *
from src.tune import tune_model
from src.utils import set_seed, save_results

def run_experiment(dataset_name='adult', data_path='data/raw/adult.data', seeds=[42, 123, 2024]):
    X, y = load_adult_data(data_path)
    num_cols, cat_cols = get_feature_types(X)
    
    all_results = []
    for seed in seeds:
        print(f"\n========== Seed {seed} ==========")
        set_seed(seed)
        (X_train, y_train), (X_val, y_val), (X_test, y_test) = split_data(X, y, random_state=seed)
        
        # Tree preprocessing
        X_train_tree, X_val_tree, X_test_tree, _ = preprocess_for_tree(X_train, X_val, X_test, cat_cols)
        # Linear preprocessing
        X_train_lr, X_val_lr, X_test_lr, _, _ = preprocess_for_linear(X_train, X_val, X_test, num_cols, cat_cols)
        # DL preprocessing
        (X_num_train, X_cat_train), (X_num_val, X_cat_val), (X_num_test, X_cat_test), _, cat_cardinalities = preprocess_for_dl(
            X_train, X_val, X_test, num_cols, cat_cols)
        
        models = ['xgboost', 'lightgbm', 'logistic', 'tabnet', 'ft_transformer']
        for model_name in models:
            print(f"  Running {model_name}...")
            if model_name in ['xgboost', 'lightgbm', 'logistic', 'tabnet']:
                # 调参次数改为 2 以便快速测试（正式运行时改为 30）
                best_params = tune_model(model_name, (X_train_tree, y_train), (X_val_tree, y_val), n_trials=2)
                if model_name == 'xgboost':
                    model = train_xgboost(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
                elif model_name == 'lightgbm':
                    model = train_lightgbm(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
                elif model_name == 'logistic':
                    model = train_logistic(X_train_lr, y_train, X_val_lr, y_val, best_params, seed)
                    X_test_model = X_test_lr
                elif model_name == 'tabnet':
                    model = train_tabnet(X_train_tree, y_train, X_val_tree, y_val, best_params, seed)
                    X_test_model = X_test_tree
            else:  # ft_transformer
                best_params = tune_model(model_name, ((X_num_train, X_cat_train), y_train),
                                         ((X_num_val, X_cat_val), y_val),
                                         cat_cardinalities=cat_cardinalities, n_trials=2)
                model = train_ft_transformer(X_num_train, X_cat_train, y_train,
                                             X_num_val, X_cat_val, y_val,
                                             cat_cardinalities, best_params, seed)
                X_test_model = (X_num_test, X_cat_test)
            
            eval_metrics = evaluate_model(model, X_test_model, y_test, model_name, cat_cardinalities, X_cat_test if model_name=='ft_transformer' else None)
            eval_metrics['model'] = model_name
            eval_metrics['seed'] = seed
            eval_metrics['best_params'] = best_params
            all_results.append(eval_metrics)
            
            os.makedirs(f'results/models/{dataset_name}/seed{seed}', exist_ok=True)
            if model_name == 'tabnet':
                model.save_model(f'results/models/{dataset_name}/seed{seed}/{model_name}.zip')
            elif model_name == 'ft_transformer':
                torch.save(model.state_dict(), f'results/models/{dataset_name}/seed{seed}/{model_name}.pt')
            else:
                joblib.dump(model, f'results/models/{dataset_name}/seed{seed}/{model_name}.pkl')
    
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(f'results/{dataset_name}_results.csv', index=False)
    print("\nAll experiments completed. Results saved to results/")
    return results_df

if __name__ == '__main__':
    run_experiment()

Overwriting src/run_experiments.py


In [32]:
from src.run_experiments import run_experiment
results = run_experiment()


========== Seed 42 ==========


[I 2026-04-05 04:05:54,228] A new study created in memory with name: no-name-650e91f0-18cf-488d-a53b-12acde62c47c


  Running xgboost...


[I 2026-04-05 04:05:58,680] Trial 0 finished with value: 0.914592725773419 and parameters: {'n_estimators': 250, 'max_depth': 10, 'learning_rate': 0.1205712628744377, 'subsample': 0.8394633936788146, 'colsample_bytree': 0.6624074561769746, 'reg_alpha': 0.004207053950287938, 'reg_lambda': 0.0017073967431528124}. Best is trial 0 with value: 0.914592725773419.
[I 2026-04-05 04:06:03,271] Trial 1 finished with value: 0.9124238380313044 and parameters: {'n_estimators': 447, 'max_depth': 7, 'learning_rate': 0.11114989443094977, 'subsample': 0.608233797718321, 'colsample_bytree': 0.9879639408647978, 'reg_alpha': 2.1368329072358767, 'reg_lambda': 0.0070689749506246055}. Best is trial 0 with value: 0.914592725773419.
[I 2026-04-05 04:06:07,340] A new study created in memory with name: no-name-fc283360-85cf-4a7f-b527-36d473a1ae27


  Running lightgbm...
[LightGBM] [Info] Number of positive: 4504, number of negative: 13592
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001525 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 664
[LightGBM] [Info] Number of data points in the train set: 18096, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.248895 -> initscore=-1.104515
[LightGBM] [Info] Start training from score -1.104515
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with pos

...

[I 2026-04-05 04:20:59,508] Trial 0 finished with value: 0.8982458024774441 and parameters: {'n_d': 29, 'n_a': 62, 'n_steps': 8, 'gamma': 1.5986584841970366, 'lambda_sparse': 2.0513382630874486e-05}. Best is trial 0 with value: 0.8982458024774441.
c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.64485 |  0:00:07s
epoch 1  | loss: 0.46928 |  0:00:14s
epoch 2  | loss: 0.44807 |  0:00:21s
epoch 3  | loss: 0.45836 |  0:00:27s
epoch 4  | loss: 0.42667 |  0:00:34s
epoch 5  | loss: 0.43753 |  0:00:41s
epoch 6  | loss: 0.39333 |  0:00:48s
epoch 7  | loss: 0.38502 |  0:00:55s
epoch 8  | loss: 0.38269 |  0:01:02s
epoch 9  | loss: 0.38336 |  0:01:09s
epoch 10 | loss: 0.37963 |  0:01:16s
epoch 11 | loss: 0.37562 |  0:01:22s
epoch 12 | loss: 0.38703 |  0:01:29s
epoch 13 | loss: 0.37665 |  0:01:36s
epoch 14 | loss: 0.37104 |  0:01:43s
epoch 15 | loss: 0.37595 |  0:01:50s
epoch 16 | loss: 0.37689 |  0:01:57s
epoch 17 | loss: 0.37858 |  0:02:05s
epoch 18 | loss: 0.38375 |  0:02:12s
epoch 19 | loss: 0.38163 |  0:02:19s
epoch 20 | loss: 0.38539 |  0:02:26s
epoch 21 | loss: 0.37684 |  0:02:33s
epoch 22 | loss: 0.37452 |  0:02:40s
epoch 23 | loss: 0.37739 |  0:02:47s
epoch 24 | loss: 0.37679 |  0:02:54s
epoch 25 | loss: 0.37155 |  0:03:01s
epoch 26 | loss: 0.36845 |  0:03:08s
e

[I 2026-04-05 04:32:01,328] Trial 1 finished with value: 0.8974750358603742 and parameters: {'n_d': 16, 'n_a': 11, 'n_steps': 9, 'gamma': 1.6011150117432087, 'lambda_sparse': 0.0002607024758370766}. Best is trial 0 with value: 0.8982458024774441.
c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.67677 |  0:00:08s
epoch 1  | loss: 0.44755 |  0:00:15s
epoch 2  | loss: 0.42406 |  0:00:24s
epoch 3  | loss: 0.41332 |  0:00:33s
epoch 4  | loss: 0.38955 |  0:00:41s
epoch 5  | loss: 0.38687 |  0:00:49s
epoch 6  | loss: 0.39274 |  0:00:58s
epoch 7  | loss: 0.38712 |  0:01:06s
epoch 8  | loss: 0.38523 |  0:01:14s
epoch 9  | loss: 0.3827  |  0:01:22s
epoch 10 | loss: 0.39115 |  0:01:31s
epoch 11 | loss: 0.37809 |  0:01:39s
epoch 12 | loss: 0.37871 |  0:01:47s
epoch 13 | loss: 0.36845 |  0:01:55s
epoch 14 | loss: 0.36285 |  0:02:04s
epoch 15 | loss: 0.35998 |  0:02:12s
epoch 16 | loss: 0.36017 |  0:02:20s
epoch 17 | loss: 0.35684 |  0:02:28s
epoch 18 | loss: 0.35802 |  0:02:35s
epoch 19 | loss: 0.35326 |  0:02:44s
epoch 20 | loss: 0.35599 |  0:02:52s
epoch 21 | loss: 0.35926 |  0:03:00s
epoch 22 | loss: 0.3619  |  0:03:07s
epoch 23 | loss: 0.35979 |  0:03:16s
epoch 24 | loss: 0.36111 |  0:03:24s
epoch 25 | loss: 0.37128 |  0:03:32s
epoch 26 | loss: 0.37127 |  0:03:40s
e

KeyError: 0

In [33]:
import os
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
import time
import joblib
from src.preprocess import *
from src.train import train_xgboost, train_lightgbm, train_logistic, train_tabnet, train_ft_transformer, evaluate_model
from src.utils import set_seed

# 简化版实验：1个种子，默认参数（无调参）
set_seed(42)
X, y = load_adult_data()
num_cols, cat_cols = get_feature_types(X)
(X_train, y_train), (X_val, y_val), (X_test, y_test) = split_data(X, y, random_state=42)

# 预处理
X_train_tree, X_val_tree, X_test_tree, _ = preprocess_for_tree(X_train, X_val, X_test, cat_cols)
X_train_lr, X_val_lr, X_test_lr, _, _ = preprocess_for_linear(X_train, X_val, X_test, num_cols, cat_cols)
(X_num_train, X_cat_train), (X_num_val, X_cat_val), (X_num_test, X_cat_test), _, cat_cardinalities = preprocess_for_dl(
    X_train, X_val, X_test, num_cols, cat_cols)

results = []
models = ['xgboost', 'lightgbm', 'logistic', 'tabnet', 'ft_transformer']
default_params = {
    'xgboost': {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1},
    'lightgbm': {'n_estimators': 100, 'max_depth': 6, 'learning_rate': 0.1},
    'logistic': {'C': 1.0},
    'tabnet': {'n_d': 8, 'n_a': 8, 'n_steps': 3},
    'ft_transformer': {'embedding_dim': 64, 'n_blocks': 3, 'n_heads': 8, 'ff_dim': 128, 'dropout': 0.1, 'lr': 1e-3, 'batch_size': 256, 'epochs': 20}
}

for model_name in models:
    print(f"\nTraining {model_name} with default params...")
    try:
        if model_name == 'xgboost':
            model = train_xgboost(X_train_tree, y_train, X_val_tree, y_val, default_params[model_name], seed=42)
            X_test_ready = X_test_tree
        elif model_name == 'lightgbm':
            model = train_lightgbm(X_train_tree, y_train, X_val_tree, y_val, default_params[model_name], seed=42)
            X_test_ready = X_test_tree
        elif model_name == 'logistic':
            model = train_logistic(X_train_lr, y_train, X_val_lr, y_val, default_params[model_name], seed=42)
            X_test_ready = X_test_lr
        elif model_name == 'tabnet':
            model = train_tabnet(X_train_tree, y_train, X_val_tree, y_val, default_params[model_name], seed=42)
            X_test_ready = X_test_tree
        elif model_name == 'ft_transformer':
            model = train_ft_transformer(X_num_train, X_cat_train, y_train,
                                         X_num_val, X_cat_val, y_val,
                                         cat_cardinalities, default_params[model_name], seed=42)
            X_test_ready = (X_num_test, X_cat_test)
        # 评估
        eval_metrics = evaluate_model(model, X_test_ready, y_test, model_name, cat_cardinalities, X_cat_test if model_name=='ft_transformer' else None)
        eval_metrics['model'] = model_name
        eval_metrics['seed'] = 42
        results.append(eval_metrics)
        print(f"  AUC: {eval_metrics['auc']:.4f}, Acc: {eval_metrics['accuracy']:.4f}, F1: {eval_metrics['f1']:.4f}")
    except Exception as e:
        print(f"  Failed: {e}")

# 保存结果
results_df = pd.DataFrame(results)
os.makedirs('results', exist_ok=True)
results_df.to_csv('results/adult_results_quick.csv', index=False)
print("\n快速实验结果已保存到 results/adult_results_quick.csv")
print(results_df)


Training xgboost with default params...
  AUC: 0.9222, Acc: 0.8624, F1: 0.6955

Training lightgbm with default params...
[LightGBM] [Info] Number of positive: 4504, number of negative: 13592
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002482 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 664
[LightGBM] [Info] Number of data points in the train set: 18096, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.248895 -> initscore=-1.104515
[LightGBM] [Info] Start training from score -1.104515
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] N

c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")
c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.51076 |  0:00:02s
epoch 1  | loss: 0.43061 |  0:00:04s
epoch 2  | loss: 0.41351 |  0:00:07s
epoch 3  | loss: 0.39418 |  0:00:10s
epoch 4  | loss: 0.38152 |  0:00:13s
epoch 5  | loss: 0.37313 |  0:00:15s
epoch 6  | loss: 0.36492 |  0:00:18s
epoch 7  | loss: 0.35705 |  0:00:21s
epoch 8  | loss: 0.35645 |  0:00:24s
epoch 9  | loss: 0.35258 |  0:00:27s
epoch 10 | loss: 0.3499  |  0:00:34s
epoch 11 | loss: 0.34981 |  0:00:36s
epoch 12 | loss: 0.34482 |  0:00:40s
epoch 13 | loss: 0.34488 |  0:00:44s
epoch 14 | loss: 0.34237 |  0:00:47s
epoch 15 | loss: 0.34328 |  0:00:49s
epoch 16 | loss: 0.33691 |  0:00:52s
epoch 17 | loss: 0.34023 |  0:00:55s
epoch 18 | loss: 0.33931 |  0:00:58s
epoch 19 | loss: 0.33851 |  0:01:01s
epoch 20 | loss: 0.33712 |  0:01:04s
epoch 21 | loss: 0.332   |  0:01:07s
epoch 22 | loss: 0.32994 |  0:01:10s
epoch 23 | loss: 0.33389 |  0:01:13s
epoch 24 | loss: 0.33832 |  0:01:16s
epoch 25 | loss: 0.33315 |  0:01:18s
epoch 26 | loss: 0.33895 |  0:01:21s
e

In [38]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score
from pytorch_tabnet.tab_model import TabNetClassifier
import time
import tempfile
import shutil
import os

# ---------- 数据加载与预处理 ----------
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data'
column_names = [
    'age', 'workclass', 'fnlwgt', 'education', 'education_num',
    'marital_status', 'occupation', 'relationship', 'race', 'sex',
    'capital_gain', 'capital_loss', 'hours_per_week', 'native_country', 'income'
]
df = pd.read_csv(url, names=column_names, na_values='?', skipinitialspace=True)
df = df.dropna().reset_index(drop=True)
y = (df['income'] == '>50K').astype(int).values
X = df.drop('income', axis=1)

# 特征类型
num_cols = ['age', 'fnlwgt', 'education_num', 'capital_gain', 'capital_loss', 'hours_per_week']
cat_cols = [c for c in X.columns if c not in num_cols]

# 划分
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)

# 数值特征预处理
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

def preprocess_num(X, fit=False):
    if fit:
        X_num = num_imputer.fit_transform(X[num_cols])
        X_num = scaler.fit_transform(X_num)
    else:
        X_num = num_imputer.transform(X[num_cols])
        X_num = scaler.transform(X[num_cols])
    return X_num.astype(np.float32)

# 类别特征预处理
cat_encoders = {}
def preprocess_cat(X, fit=False):
    X_cat = np.zeros((len(X), len(cat_cols)), dtype=np.int64)
    for i, col in enumerate(cat_cols):
        if fit:
            le = LabelEncoder()
            X_cat[:, i] = le.fit_transform(X[col].astype(str))
            cat_encoders[col] = le
        else:
            le = cat_encoders[col]
            X_cat[:, i] = le.transform(X[col].astype(str))
    return X_cat

X_num_train = preprocess_num(X_train, fit=True)
X_cat_train = preprocess_cat(X_train, fit=True)
X_num_val = preprocess_num(X_val, fit=False)
X_cat_val = preprocess_cat(X_val, fit=False)
X_num_test = preprocess_num(X_test, fit=False)
X_cat_test = preprocess_cat(X_test, fit=False)

# TabNet 输入：合并数值和类别
X_train_tab = np.hstack([X_num_train, X_cat_train])
X_val_tab = np.hstack([X_num_val, X_cat_val])
X_test_tab = np.hstack([X_num_test, X_cat_test])

# ---------- 1. TabNet ----------
print("Training TabNet...")
model_tab = TabNetClassifier(seed=42, n_d=8, n_a=8, n_steps=3)
model_tab.fit(
    X_train_tab, y_train,
    eval_set=[(X_val_tab, y_val)],
    eval_metric=['auc'],
    max_epochs=20,
    patience=5,
    batch_size=256,
    virtual_batch_size=128
)
y_pred_proba_tab = model_tab.predict_proba(X_test_tab)[:, 1]
y_pred_binary_tab = (y_pred_proba_tab >= 0.5).astype(int)
auc_tab = roc_auc_score(y_test, y_pred_proba_tab)
acc_tab = accuracy_score(y_test, y_pred_binary_tab)
f1_tab = f1_score(y_test, y_pred_binary_tab)

# 推理延迟
start = time.perf_counter()
for _ in range(10):
    _ = model_tab.predict_proba(X_test_tab[:1000])[:, 1]
latency_tab = (time.perf_counter() - start) / (10 * 1000) * 1000

# 模型大小（使用临时目录）
tmp_dir = tempfile.mkdtemp()
try:
    model_tab.save_model(tmp_dir)  # save_model 期望目录
    size_tab = sum(os.path.getsize(os.path.join(dirpath, f)) for dirpath, _, filenames in os.walk(tmp_dir) for f in filenames) / (1024 * 1024)
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

print(f"TabNet - AUC: {auc_tab:.4f}, Acc: {acc_tab:.4f}, F1: {f1_tab:.4f}, Latency: {latency_tab:.4f} ms/sample, Size: {size_tab:.4f} MB")

# ---------- 2. FT-Transformer ----------
class FeatureTokenizer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64):
        super().__init__()
        self.n_num_features = n_num_features
        if n_num_features > 0:
            self.num_linear = nn.Linear(n_num_features, embedding_dim)
        else:
            self.num_linear = None
        self.cat_embeddings = nn.ModuleList([nn.Embedding(card, embedding_dim) for card in cat_cardinalities])
    def forward(self, x_num, x_cat):
        tokens = []
        if self.num_linear is not None:
            tokens.append(self.num_linear(x_num))
        for i, emb in enumerate(self.cat_embeddings):
            tokens.append(emb(x_cat[:, i].long()))
        return torch.stack(tokens, dim=1)

class FTTransformer(nn.Module):
    def __init__(self, n_num_features, cat_cardinalities, embedding_dim=64, n_blocks=3, n_heads=8, ff_dim=128, dropout=0.1, output_dim=1):
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_num_features, cat_cardinalities, embedding_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, embedding_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=embedding_dim, nhead=n_heads, dim_feedforward=ff_dim, dropout=dropout, activation='gelu', batch_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_blocks)
        self.output_layer = nn.Linear(embedding_dim, output_dim)
    def forward(self, x_num, x_cat):
        tokens = self.tokenizer(x_num, x_cat)
        cls = self.cls_token.expand(tokens.size(0), -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        tokens = self.transformer(tokens)
        cls_out = tokens[:, 0, :]
        return self.output_layer(cls_out).squeeze(-1)

n_num = X_num_train.shape[1]
cat_cardinalities = [len(cat_encoders[col].classes_) for col in cat_cols]

X_num_train_t = torch.tensor(X_num_train, dtype=torch.float32)
X_cat_train_t = torch.tensor(X_cat_train, dtype=torch.long)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_num_val_t = torch.tensor(X_num_val, dtype=torch.float32)
X_cat_val_t = torch.tensor(X_cat_val, dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.float32)
X_num_test_t = torch.tensor(X_num_test, dtype=torch.float32)
X_cat_test_t = torch.tensor(X_cat_test, dtype=torch.long)

train_dataset = TensorDataset(X_num_train_t, X_cat_train_t, y_train_t)
val_dataset = TensorDataset(X_num_val_t, X_cat_val_t, y_val_t)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_ft = FTTransformer(n_num, cat_cardinalities, embedding_dim=64, n_blocks=3, n_heads=8, ff_dim=128, dropout=0.1).to(device)
optimizer = optim.Adam(model_ft.parameters(), lr=1e-3)
criterion = nn.BCEWithLogitsLoss()

best_val_auc = 0
patience = 5
epochs_no_improve = 0
print("\nTraining FT-Transformer...")
for epoch in range(30):
    model_ft.train()
    total_loss = 0
    for x_num, x_cat, y in train_loader:
        x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model_ft(x_num, x_cat)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    # validation
    model_ft.eval()
    y_pred_val = []
    y_true_val = []
    with torch.no_grad():
        for x_num, x_cat, y in val_loader:
            x_num, x_cat, y = x_num.to(device), x_cat.to(device), y.to(device)
            logits = model_ft(x_num, x_cat)
            probs = torch.sigmoid(logits)
            y_pred_val.append(probs.cpu().numpy())
            y_true_val.append(y.cpu().numpy())
    y_pred_val = np.concatenate(y_pred_val)
    y_true_val = np.concatenate(y_true_val)
    val_auc = roc_auc_score(y_true_val, y_pred_val)
    print(f"Epoch {epoch+1:2d} | loss: {total_loss/len(train_loader):.5f} | val_auc: {val_auc:.4f}")
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        epochs_no_improve = 0
        best_model_state = {k: v.cpu() for k, v in model_ft.state_dict().items()}
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping")
            break
model_ft.load_state_dict(best_model_state)

# 测试评估
model_ft.eval()
test_dataset = TensorDataset(X_num_test_t, X_cat_test_t)
test_loader = DataLoader(test_dataset, batch_size=256)
y_pred_test = []
start = time.perf_counter()
with torch.no_grad():
    for x_num, x_cat in test_loader:
        x_num, x_cat = x_num.to(device), x_cat.to(device)
        logits = model_ft(x_num, x_cat)
        probs = torch.sigmoid(logits)
        y_pred_test.append(probs.cpu().numpy())
latency_ft = (time.perf_counter() - start) / len(X_num_test) * 1000
y_pred_test = np.concatenate(y_pred_test)
y_pred_binary_ft = (y_pred_test >= 0.5).astype(int)
auc_ft = roc_auc_score(y_test, y_pred_test)
acc_ft = accuracy_score(y_test, y_pred_binary_ft)
f1_ft = f1_score(y_test, y_pred_binary_ft)

# 模型大小（使用临时目录）
tmp_dir = tempfile.mkdtemp()
try:
    tmp_path = os.path.join(tmp_dir, 'model.pt')
    torch.save(model_ft.state_dict(), tmp_path)
    size_ft = os.path.getsize(tmp_path) / (1024 * 1024)
finally:
    shutil.rmtree(tmp_dir, ignore_errors=True)

print(f"\nFT-Transformer - AUC: {auc_ft:.4f}, Acc: {acc_ft:.4f}, F1: {f1_ft:.4f}, Latency: {latency_ft:.4f} ms/sample, Size: {size_ft:.4f} MB")

# 汇总所有模型结果（包括之前成功的三个）
all_results = pd.DataFrame([
    {'model': 'XGBoost', 'auc': 0.922219, 'accuracy': 0.862423, 'f1': 0.695525, 'latency_ms': 0.04626, 'size_mb': 0.295},
    {'model': 'LightGBM', 'auc': 0.922505, 'accuracy': 0.863418, 'f1': 0.701233, 'latency_ms': 0.01877, 'size_mb': 0.322},
    {'model': 'Logistic', 'auc': 0.901916, 'accuracy': 0.846842, 'f1': 0.662034, 'latency_ms': 0.00097, 'size_mb': 0.0016},
    {'model': 'TabNet', 'auc': auc_tab, 'accuracy': acc_tab, 'f1': f1_tab, 'latency_ms': latency_tab, 'size_mb': size_tab},
    {'model': 'FT-Transformer', 'auc': auc_ft, 'accuracy': acc_ft, 'f1': f1_ft, 'latency_ms': latency_ft, 'size_mb': size_ft}
])
print("\n=== 所有模型结果汇总 ===")
print(all_results.to_string(index=False))

os.makedirs('results', exist_ok=True)
all_results.to_csv('results/adult_all_models.csv', index=False)
print("\n结果已保存到 results/adult_all_models.csv")

c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


Training TabNet...
epoch 0  | loss: 0.45727 | val_0_auc: 0.70761 |  0:00:13s
epoch 1  | loss: 0.38589 | val_0_auc: 0.84848 |  0:00:24s
epoch 2  | loss: 0.36182 | val_0_auc: 0.86502 |  0:00:35s
epoch 3  | loss: 0.36142 | val_0_auc: 0.88227 |  0:00:43s
epoch 4  | loss: 0.35576 | val_0_auc: 0.88475 |  0:00:54s
epoch 5  | loss: 0.35591 | val_0_auc: 0.88858 |  0:01:02s
epoch 6  | loss: 0.3492  | val_0_auc: 0.89207 |  0:01:09s
epoch 7  | loss: 0.34903 | val_0_auc: 0.89535 |  0:01:15s
epoch 8  | loss: 0.34827 | val_0_auc: 0.8955  |  0:01:21s
epoch 9  | loss: 0.35174 | val_0_auc: 0.89734 |  0:01:26s
epoch 10 | loss: 0.34774 | val_0_auc: 0.89542 |  0:01:30s
epoch 11 | loss: 0.34404 | val_0_auc: 0.90214 |  0:01:36s
epoch 12 | loss: 0.34505 | val_0_auc: 0.89688 |  0:01:41s
epoch 13 | loss: 0.34318 | val_0_auc: 0.8994  |  0:01:46s
epoch 14 | loss: 0.33884 | val_0_auc: 0.90298 |  0:01:51s
epoch 15 | loss: 0.34107 | val_0_auc: 0.89727 |  0:01:56s
epoch 16 | loss: 0.33893 | val_0_auc: 0.89537 |  0:02

c:\Users\king\Desktop\tabular_dl_project\venv\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


Successfully saved model at C:\Users\king\AppData\Local\Temp\tmpb1fv26yd.zip
TabNet - AUC: 0.9010, Acc: 0.8420, F1: 0.6367, Latency: 0.1176 ms/sample, Size: 0.0000 MB

Training FT-Transformer...
Epoch  1 | loss: 0.41293 | val_auc: 0.9035
Epoch  2 | loss: 0.32859 | val_auc: 0.9106
Epoch  3 | loss: 0.31943 | val_auc: 0.9125
Epoch  4 | loss: 0.31772 | val_auc: 0.9127
Epoch  5 | loss: 0.31410 | val_auc: 0.9129
Epoch  6 | loss: 0.31115 | val_auc: 0.9127
Epoch  7 | loss: 0.30771 | val_auc: 0.9144
Epoch  8 | loss: 0.30373 | val_auc: 0.9140
Epoch  9 | loss: 0.30217 | val_auc: 0.9130
Epoch 10 | loss: 0.30002 | val_auc: 0.9132
Epoch 11 | loss: 0.29732 | val_auc: 0.9110
Epoch 12 | loss: 0.29650 | val_auc: 0.9118
Early stopping

FT-Transformer - AUC: 0.9097, Acc: 0.8502, F1: 0.6971, Latency: 0.5888 ms/sample, Size: 0.4232 MB

=== 所有模型结果汇总 ===
         model      auc  accuracy       f1  latency_ms  size_mb
       XGBoost 0.922219  0.862423 0.695525    0.046260 0.295000
      LightGBM 0.922505  0.86